In [1]:
import argparse
import logging
import os, sys
import torch
import numpy as np
import random
import json

# To set deterministic behaviour:
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # or ':16:8'
sys.path.append('/Data_large/marine/PythonProjects/MMDET/MyConfigs')

from mmengine.config import Config, DictAction
from mmengine.logging import print_log
from mmengine.registry import RUNNERS
from mmengine.runner import Runner
from mmdet.evaluation import DumpDetResults

from mmdet.utils import setup_cache_size_limit_of_dynamo

import logging

def set_logger(workdir):
    """
    Sets up a logger that logs exclusively to a file.

    Parameters:
    -----------
    workdir : str
        The directory where the log file ('executor.log') will be saved.

    Returns:
    --------
    logging.Logger
        Configured logger that writes logs to a file.
    """
    # Set up the logger
    logger = logging.getLogger(__name__)
    logger.setLevel(logging.DEBUG)

    # Remove any existing handlers
    logger.handlers = []

    # Create a file handler that logs to 'executor.log'
    file_handler = logging.FileHandler(f'{workdir}/MyNetwork.log')
    file_handler.setLevel(logging.DEBUG)

    # Create a formatter and set it for the handler
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    file_handler.setFormatter(formatter)

    # Add the file handler to the logger
    logger.addHandler(file_handler)

    return logger

def init_cfg():
    base_folder = '/Data_large/marine/PythonProjects/MMDET/MyConfigs'
    cfg = Config.fromfile(f'{base_folder}/Venus_b5/vfnet_r18.py')
    return cfg


def set_seed(seed):
    # Set the seed for generating random numbers in PyTorch
    torch.manual_seed(seed)
    # If using GPUs, ensure that the random numbers are generated the same way
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # if you are using multi-GPU.
    
    # Set the seed for generating random numbers in Python
    random.seed(seed)
    
    # Set the seed for generating random numbers in numpy
    np.random.seed(seed)
    
    # Ensure deterministic behavior by setting the flag
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
    # Optionally, set environment variables to ensure reproducibility
    os.environ['PYTHONHASHSEED'] = str(seed)

SEED = 42
setup_cache_size_limit_of_dynamo()

# Deterministic Behaviour setting:
set_seed(SEED)
cfg = init_cfg()
cfg.randomness = dict(
    seed = SEED, # 41 72 18
    diff_rank_seed=True,
    # deterministic=True
)

### Custom

In [2]:
SENSOR = 'VENUS'

BAND_SEL = [3,4,7] # Selecting the Band for Venus or Sentinel
resize = 2048
BS = 1
LR = 0.001
random_crop = None
MAX_EPOCHS = 20 #
selOpt = 'SGD'

In [3]:
AMP = False

# Normalization:
MEANS=[200,154,92,63] if SENSOR == 'SENTINEL' else [158.69588,124.42161,109.27108,105.380424,88.40926,98.93067,88.819916,94.20678,103.540764,111.64337,122.92817,79.31501] 
STD=[22,24,22,60] if SENSOR == 'SENTINEL' else [34.95446,46.282494,56.252197,55.741932,64.54027,59.59095,69.65824,68.40028,77.930405,103.4634,105.30468,65.8369]

indexCorrect = {2:0, 3:1, 4:2, 8:3} if SENSOR == 'SENTINEL' else {i:i-1 for i in range(1, 13, 1)}
MEAN_VALS = [MEANS[indexCorrect[x]] for x in BAND_SEL]
STD_VALS = [STD[indexCorrect[x]] for x in BAND_SEL]
# Resizing:
IMG_SIZE = resize

# Annotations:
base_annot = '/Data_large/marine/Datasets/VDS2Raw/annotations' if SENSOR == 'SENTINEL' else '/Data_large/marine/Datasets/VENuS/annotations/perfect'
ann_file = {'Train': f'{base_annot}/train__band_{BAND_SEL[0]}.json',
            'Val': f'{base_annot}/val__band_{BAND_SEL[0]}.json',
            'Test': f'{base_annot}/test__band_{BAND_SEL[0]}.json',
            }

## Dataloading Directories:
data_root = '/Data_large/marine/Datasets/VDS2Raw/' if SENSOR == 'SENTINEL' else '/Data_large/marine/Datasets/VENuS' # where the images are stored
data_prefix = f'imgs/' if SENSOR == 'SENTINEL' else f'ds_L0/perfect/'


optimizers =  {'SGD':{'type': 'OptimWrapper', 'optimizer': {'type': 'SGD', 'lr': LR, 'momentum': 0.9, 'weight_decay': 0.0001}},
            'Adam':{'type': 'OptimWrapper', 'optimizer': {'type': 'Adam', 'lr': LR, 'weight_decay': 0.0001}},
            'AdamW':{'type': 'OptimWrapper', 'optimizer': {'type': 'Adam', 'lr': LR, 'weight_decay': 0.0001}},}


## Savedir:
bandsNames = ''.join([f'_b{x}' for x in BAND_SEL])
singleMulti = 'Multi' if len(BAND_SEL) > 1 else 'Single'
# WORKDIR:
kMode = {'SENTINEL':'Sentinel', 'VENUS':'VENuS'}
workdir = f'/Data_large/marine/PythonProjects/MMDET/checkpoints/{kMode[SENSOR]}/MyNet/{singleMulti}/perfect{bandsNames}/{SEED}_BS_{BS}_LR_{LR}_ME_{MAX_EPOCHS}_OPT_{selOpt}'
cfg.work_dir = workdir


#### AMP:
# enable automatic-mixed-precision training
if AMP is True:
    optim_wrapper = cfg.optim_wrapper.type
    if optim_wrapper == 'AmpOptimWrapper':
        print_log(
            'AMP training is already enabled in your config.',
            logger='current',
            level=logging.WARNING)
    else:
        assert optim_wrapper == 'OptimWrapper', (
            '`--amp` is only supported when the optimizer wrapper type is '
            f'`OptimWrapper` but got {optim_wrapper}.')
        cfg.optim_wrapper.type = 'AmpOptimWrapper'
        cfg.optim_wrapper.loss_scale = 'dynamic'

# Dataloader:
cfg.model.data_preprocessor = dict(
    mean=[float(x) for x in MEAN_VALS],
    pad_size_divisor=1,
    std=[float(x) for x in STD_VALS],
    type='MyPrePro')

# Model Inputs:
cfg.model.backbone.in_channels = len(BAND_SEL)

# Annotation file:
cfg.train_dataloader.dataset.ann_file = ann_file['Train']
cfg.train_dataloader.dataset.data_prefix = {'img':data_prefix}
cfg.train_dataloader.dataset.data_root = data_root

cfg.val_dataloader.dataset.ann_file = ann_file['Val']
cfg.val_dataloader.dataset.data_prefix = {'img':data_prefix}
cfg.val_dataloader.dataset.data_root = data_root

#       Evaluators:
cfg.val_evaluator = dict(
    ann_file=ann_file['Val'],
    backend_args=None,
    format_only=False,
    metric='bbox',
    type='CocoMetric')

# Pipeline:
# Hook for custom loader:
loadCorrect = {2:1, 3:2, 4:3, 8:4} if SENSOR == 'SENTINEL' else {i:i for i in range(1, 13, 1)}
BAND_SEL_LOAD = [loadCorrect[x] for x in BAND_SEL]

cfg.train_dataloader.dataset.pipeline[0] = {'type': 'SelBandLoader', 'to_float32': True, 'bands_list': BAND_SEL_LOAD}
cfg.val_dataloader.dataset.pipeline[0] = {'type': 'SelBandLoader', 'to_float32': True, 'bands_list': BAND_SEL_LOAD}

cfg.train_dataloader.dataset.pipeline[3] = {'type': 'Resize', 'scale': (IMG_SIZE, IMG_SIZE), 'keep_ratio': False}
cfg.val_dataloader.dataset.pipeline[2] = {'type': 'Resize', 'scale': (IMG_SIZE, IMG_SIZE), 'keep_ratio': False}

# Adding random crop to the pipeline. TODO: training with decreasing size
if random_crop is not None:
    assert isinstance(random_crop, int), 'RandomCrop Error: single dimension must be specified. E.g. 224'
    # insert random crop: 
    rc = dict(type='RandomCrop', crop_size=(random_crop, random_crop))
    cfg.train_dataloader.dataset.pipeline.insert(3, rc)
    cfg.val_dataloader.dataset.pipeline.insert(2, rc)
    

# Training params:
cfg.train_dataloader.batch_size = BS
cfg.train_cfg = {'type': 'EpochBasedTrainLoop', 'max_epochs': MAX_EPOCHS, 'val_interval': 1}

# TODO: implement stages as in: https://github.com/open-mmlab/mmdetection/blob/cfd5d3a985b0249de009b67d04f37263e11cdf3d/configs/rtmdet/rtmdet_x_p6_4xb8-300e_coco.py#L78
# lr_config = dict(policy='poly', power=0.9, min_lr=1e-4, by_epoch=False)

cfg.optim_wrapper = optimizers[selOpt]

# TODO: reset correct scheduler
cfg.param_scheduler = [{'type': 'LinearLR',
                        'start_factor': 0.001,
                        'by_epoch': True,
                        'begin': 0,
                        'end': 1},
                        {'type': 'MultiStepLR',
                        'begin': 0,
                        'end': MAX_EPOCHS//5,
                        'by_epoch': True,
                        'milestones': [MAX_EPOCHS//4, MAX_EPOCHS//3, MAX_EPOCHS//2],
                        'gamma': 0.75}, 
                        {# use cosine lr scheduler
                        'type':'CosineAnnealingLR',
                        'eta_min':LR * 0.05,
                        'begin':MAX_EPOCHS//2,
                        'end':MAX_EPOCHS,
                        'T_max':MAX_EPOCHS//1.5,
                        'by_epoch':True,
                        'convert_to_iter_based':True,}
                        ]

#### Test Config hooks:
default_hooks = cfg.default_hooks
if 'visualization' in default_hooks:
    visualization_hook = default_hooks['visualization']
    # Turn on visualization
    visualization_hook['draw'] = False

cfg.test_dataloader = dict(
            batch_size=1,
            dataset=dict(
                ann_file=ann_file['Test'],
                data_root=data_root,
                data_prefix=dict(img=data_prefix),
                filter_cfg=dict(filter_empty_gt=True),
                metainfo=dict(classes=('vessel', ), palette=[
                    (
                        220,
                        20,
                        60,
                    ),
                ]),
                pipeline=[{'type': 'SelBandLoader', 'to_float32': True, 'bands_list': BAND_SEL_LOAD},
                    dict(type='LoadAnnotations', with_bbox=True),
                    dict(keep_ratio=False, scale=(IMG_SIZE,IMG_SIZE,), type='Resize'),
                    dict(
                        meta_keys=('img_path', 'img_id', 'seg_map_path', 
                                'height', 'width', 'instances', 'sample_idx', 
                                'img', 'img_shape', 'ori_shape', 'scale', 'scale_factor', 
                                'keep_ratio', 'homography_matrix', 'gt_bboxes', 'gt_ignore_flags', 
                                'gt_bboxes_labels'),
                        type='PackDetInputs'),
                ],
                test_mode=True,
                type='CocoDataset'),
            drop_last=False,
            num_workers=2,
            persistent_workers=True,
            sampler=dict(shuffle=False, type='DefaultSampler'))

cfg.test_evaluator = dict(
            type='CocoMetric',
            metric='bbox',
            format_only=False,
            ann_file=ann_file['Test'],
            outfile_prefix=f'{workdir}/test_results')

#### Model Customization:

In [4]:
from mmdet.models.backbones import EfficientNet, ResNet
from mmcv.cnn.bricks import ConvModule

######## BACKBONE CUSTOMIZATION:
out_indices = (2,3,4,5,)
# activations = 'Swish'
activations = 'ReLU' # Swtiched from Swish to ReLU for easy deployment 

backbone = EfficientNet(
    arch='b0',
    drop_path_rate=0.,
    out_indices=out_indices,
    frozen_stages=0,
    conv_cfg=dict(type='Conv2dAdaptivePadding'),
    norm_cfg=dict(type='BN', eps=1e-3),
    act_cfg=dict(type=activations),
    norm_eval=False,
    with_cp=False,
    init_cfg=[
        dict(type='Kaiming', layer='Conv2d'),
        dict(
            type='Constant',
            layer=['_BatchNorm', 'GroupNorm'],
            val=1)]
)


# backbone = ResNet(
#     depth=18,
#     num_stages=4,
#     out_indices=(0, 1, 2, 3),
#     style='pytorch',
#     norm_cfg=dict(type='BN', requires_grad=True),
#     norm_eval=False,
#     zero_init_residual=False,
#     frozen_stages=1,
#     init_cfg=dict(type='Pretrained', checkpoint='torchvision://resnet18')
# )

- Customization of the first layer to change the input channel from 3 to 1.
- Changed the activations from Swish to ReLU.

In [5]:
# Failed attempt to change the ConvModule:
"""in_channels = 3
mid_channels = 32
kernel_size = 3
stride = 2
conv_cfg = dict(type='Conv2dAdaptivePadding')
norm_cfg = dict(type='BN', eps=1e-3)
act_cfg = dict(type=activations)

conv1 = ConvModule(
            in_channels=in_channels,
            out_channels=mid_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=kernel_size // 2,
            conv_cfg=conv_cfg,
            norm_cfg=norm_cfg,
            act_cfg=act_cfg)
            
backbone.layers[0] = conv1            
"""

"in_channels = 3\nmid_channels = 32\nkernel_size = 3\nstride = 2\nconv_cfg = dict(type='Conv2dAdaptivePadding')\nnorm_cfg = dict(type='BN', eps=1e-3)\nact_cfg = dict(type=activations)\n\nconv1 = ConvModule(\n            in_channels=in_channels,\n            out_channels=mid_channels,\n            kernel_size=kernel_size,\n            stride=stride,\n            padding=kernel_size // 2,\n            conv_cfg=conv_cfg,\n            norm_cfg=norm_cfg,\n            act_cfg=act_cfg)\n            \nbackbone.layers[0] = conv1            \n"

- Backtesting if the model is working fine with the customizations.

In [6]:
# pass a x to the backbone to get the output
x = torch.randn(1, 3, 448, 448)
out = backbone(x)

In [7]:
backbone

EfficientNet(
  (layers): ModuleList(
    (0): ConvModule(
      (conv): Conv2dAdaptivePadding(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
      (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
      (activate): ReLU(inplace=True)
    )
    (1): Sequential(
      (0): InvertedResidual(
        (drop_path): Identity()
        (depthwise_conv): ConvModule(
          (conv): Conv2dAdaptivePadding(32, 32, kernel_size=(3, 3), stride=(1, 1), groups=32, bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
          (activate): ReLU(inplace=True)
        )
        (se): SELayer(
          (global_avgpool): AdaptiveAvgPool2d(output_size=1)
          (conv1): ConvModule(
            (conv): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (activate): ReLU(inplace=True)
          )
          (conv2): ConvModule(
            (conv): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
    

- Switchng the backbone to EfficientNetB0-Custom.

In [8]:
"""
# This fails as you cannot substitude a backbone inside a cfg ... 
cfg.model.backbone = backbone
""" 

'\n# This fails as you cannot substitude a backbone inside a cfg ... \ncfg.model.backbone = backbone\n'

#### Custom Neck:

We need to change the input channels of the Neck according to the new backbone. We gonna use a forward pass to determine the number of input channels of the Neck.


EfficientNet-B0:

In [9]:
in_channels_neck = []
for item in out:
    in_channels_neck.append(item.shape[1])
    print(item.shape)

torch.Size([1, 24, 112, 112])
torch.Size([1, 40, 56, 56])
torch.Size([1, 112, 28, 28])
torch.Size([1, 320, 14, 14])


In [10]:
from mmdet.models.necks import FPN 
from typing import List, Union

neck = FPN(
    in_channels=in_channels_neck,
    out_channels=256,
    num_outs=len(in_channels_neck),
    start_level=0,
    end_level=3,
    add_extra_convs=False,
    relu_before_extra_convs=False,
    no_norm_on_lateral=False,
    conv_cfg=None,
    norm_cfg=None,
    act_cfg=None,
    upsample_cfg=dict(mode='nearest'),
    init_cfg=dict(type='Xavier', layer='Conv2d', distribution='uniform')
)


Backtesting the neck with the new backbone.


In [11]:
import torch
in_channels = in_channels_neck

outputs = neck(out)
for i in range(len(outputs)):
    print(f'outputs[{i}].shape = {outputs[i].shape}')

outputs[0].shape = torch.Size([1, 256, 112, 112])
outputs[1].shape = torch.Size([1, 256, 56, 56])
outputs[2].shape = torch.Size([1, 256, 28, 28])
outputs[3].shape = torch.Size([1, 256, 14, 14])


## Config Update:

Update the neck with the inchannels

In [12]:
cfg.model.backbone = {
    'type': 'EfficientNet',
    'arch': 'b0',
    'drop_path_rate': 0.0,
    'out_indices': (2, 3, 4, 5),
    'frozen_stages': 0,
    'conv_cfg': {'type': 'Conv2dAdaptivePadding'},
    'norm_cfg': {'type': 'BN', 'eps': 0.001},
    'act_cfg': {'type': 'ReLU'},
    'norm_eval': False,
    'with_cp': False,
    'init_cfg': [{'type': 'Kaiming', 'layer': 'Conv2d'},
                {'type': 'Constant',
                'layer': ['_BatchNorm', 'GroupNorm'],
                'val': 1}]
}

In [13]:
cfg.model.neck = {
    'type': 'FPN',
    'in_channels': in_channels_neck,
    'out_channels': 256,
    'num_outs': len(in_channels_neck) + 1,
    'start_level': 1, # TODO; check if this is correct
    'end_level': -1,
    'add_extra_convs': 'on_output',
    'relu_before_extra_convs': True,
    'no_norm_on_lateral': False,
    'conv_cfg': None,
    'norm_cfg': None,
    'act_cfg': None,
    'upsample_cfg': dict(mode='nearest'),
    'init_cfg': dict(type='Xavier', layer='Conv2d', distribution='uniform')
}



# type='FPN',
# in_channels=[256, 512, 1024, 2048],
# out_channels=256,
# start_level=1,
# add_extra_convs='on_output',  # use P5
# num_outs=5,
# relu_before_extra_convs=True),

In [14]:
cfg.model

{'type': 'VFNet',
 'data_preprocessor': {'mean': [109.27108, 105.380424, 88.819916],
  'pad_size_divisor': 1,
  'std': [56.252197, 55.741932, 69.65824],
  'type': 'MyPrePro'},
 'backbone': {'type': 'EfficientNet',
  'arch': 'b0',
  'drop_path_rate': 0.0,
  'out_indices': (2, 3, 4, 5),
  'frozen_stages': 0,
  'conv_cfg': {'type': 'Conv2dAdaptivePadding'},
  'norm_cfg': {'type': 'BN', 'eps': 0.001},
  'act_cfg': {'type': 'ReLU'},
  'norm_eval': False,
  'with_cp': False,
  'init_cfg': [{'type': 'Kaiming', 'layer': 'Conv2d'},
   {'type': 'Constant', 'layer': ['_BatchNorm', 'GroupNorm'], 'val': 1}]},
 'neck': {'type': 'FPN',
  'in_channels': [24, 40, 112, 320],
  'out_channels': 256,
  'num_outs': 5,
  'start_level': 1,
  'end_level': -1,
  'add_extra_convs': 'on_output',
  'relu_before_extra_convs': True,
  'no_norm_on_lateral': False,
  'conv_cfg': None,
  'norm_cfg': None,
  'act_cfg': None,
  'upsample_cfg': {'mode': 'nearest'},
  'init_cfg': {'type': 'Xavier',
   'layer': 'Conv2d',
  

#### RUNNER SET

Setting the runner from cfg 

In [15]:
# build the runner from config
if 'runner_type' not in cfg:
    # build the default runner
    runner = Runner.from_cfg(cfg)
else:
    # build customized runner from the registry
    # if 'runner_type' is set in the cfg
    runner = RUNNERS.build(cfg)

08/16 09:45:20 - mmengine - INFO - 
------------------------------------------------------------
System environment:
    sys.platform: linux
    Python: 3.8.19 | packaged by conda-forge | (default, Mar 20 2024, 12:47:35) [GCC 12.3.0]
    CUDA available: True
    MUSA available: False
    numpy_random_seed: 42
    GPU 0: NVIDIA A100-SXM4-40GB
    CUDA_HOME: /usr/local/cuda-11.4
    NVCC: Cuda compilation tools, release 11.4, V11.4.152
    GCC: gcc (Ubuntu 9.4.0-1ubuntu1~20.04.2) 9.4.0
    PyTorch: 2.0.0+cu118
    PyTorch compiling details: PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v2.7.3 (Git Hash 6dbeffbae1f23cbbeae17adb7b5b13f1f37c080e)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX2
  - CUDA Runtime 11.8
  - NVCC architecture flags: -gencode;

#### RUNNER START

In [16]:
logger = set_logger(workdir)
logger.info(f'Config: {cfg.pretty_text}')
logger.info(f'Workdir: {workdir}')
logger.info(f'Random Seed: {SEED}')
logger.info(f'Band Selection: {BAND_SEL}')
logger.info(f'Band Load Selection: {BAND_SEL_LOAD}')
logger.info(f'Image Size: {IMG_SIZE}')
logger.info(f'Batch Size: {BS}')
logger.info(f'Learning Rate: {LR}')
logger.info(f'Max Epochs: {MAX_EPOCHS}')
logger.info(f'Optimizer: {selOpt}')
logger.info(f'AMP: {AMP}')
logger.info(f'Random Crop: {random_crop}')

logger.info('*** Start training ***')
runner.train()

loading annotations into memory...
Done (t=0.05s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
08/16 09:45:25 - mmengine - WARNING - "FileClient" will be deprecated in future. Please use io functions in https://mmengine.readthedocs.io/en/latest/api/fileio.html#file-io
08/16 09:45:25 - mmengine - WARNING - "HardDiskBackend" is the alias of "LocalBackend" and the former will be deprecated in future.
08/16 09:45:25 - mmengine - INFO - Checkpoints will be saved to /Data_large/marine/PythonProjects/MMDET/checkpoints/VENuS/MyNet/Multi/perfect_b3_b4_b7/42_BS_1_LR_0.001_ME_20_OPT_SGD.


/home/vessel/anaconda3/envs/openmmlab/lib/python3.8/site-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3483.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


08/16 09:45:31 - mmengine - INFO - Epoch(train)  [1][  1/140]  lr: 1.0000e-06  eta: 4:16:21  time: 5.4954  data_time: 0.9137  memory: 8382  loss: 1.4099  loss_cls: 0.0038  loss_bbox: 0.6994  loss_bbox_rf: 0.7067
08/16 09:45:31 - mmengine - INFO - Epoch(train)  [1][  2/140]  lr: 1.0000e-06  eta: 2:15:48  time: 2.9122  data_time: 0.4862  memory: 8439  loss: 1.4859  loss_cls: 0.0030  loss_bbox: 0.7356  loss_bbox_rf: 0.7473
08/16 09:45:31 - mmengine - INFO - Epoch(train)  [1][  3/140]  lr: 1.0000e-06  eta: 1:35:05  time: 2.0398  data_time: 0.3308  memory: 8439  loss: 1.3869  loss_cls: 0.0024  loss_bbox: 0.6862  loss_bbox_rf: 0.6982
08/16 09:45:32 - mmengine - INFO - Epoch(train)  [1][  4/140]  lr: 1.0000e-06  eta: 1:18:19  time: 1.6807  data_time: 0.2513  memory: 8439  loss: 1.6609  loss_cls: 0.0023  loss_bbox: 0.8253  loss_bbox_rf: 0.8334
08/16 09:45:33 - mmengine - INFO - Epoch(train)  [1][  5/140]  lr: 1.0000e-06  eta: 1:08:23  time: 1.4680  data_time: 0.2043  memory: 8439  loss: 1.5966

/home/vessel/anaconda3/envs/openmmlab/lib/python3.8/site-packages/mmengine/hooks/early_stopping_hook.py:148: UserWarning: Skip early stopping process since the evaluation results (dict_keys([])) do not include `monitor` (coco/bbox_mAP_50).
  warnings.warn(


08/16 09:47:46 - mmengine - INFO - Epoch(train)  [2][  1/140]  lr: 1.0000e-06  eta: 0:36:44  time: 0.8458  data_time: 0.0229  memory: 8439  loss: 1.4972  loss_cls: 0.0023  loss_bbox: 0.7268  loss_bbox_rf: 0.7681
08/16 09:47:47 - mmengine - INFO - Epoch(train)  [2][  2/140]  lr: 1.0000e-06  eta: 0:36:45  time: 0.8488  data_time: 0.0229  memory: 8439  loss: 1.4973  loss_cls: 0.0023  loss_bbox: 0.7271  loss_bbox_rf: 0.7679
08/16 09:47:48 - mmengine - INFO - Epoch(train)  [2][  3/140]  lr: 1.0000e-06  eta: 0:36:45  time: 0.8509  data_time: 0.0229  memory: 8439  loss: 1.5318  loss_cls: 0.0023  loss_bbox: 0.7441  loss_bbox_rf: 0.7854
08/16 09:47:49 - mmengine - INFO - Epoch(train)  [2][  4/140]  lr: 1.0000e-06  eta: 0:36:48  time: 0.8540  data_time: 0.0228  memory: 8439  loss: 1.5124  loss_cls: 0.0023  loss_bbox: 0.7349  loss_bbox_rf: 0.7752
08/16 09:47:50 - mmengine - INFO - Epoch(train)  [2][  5/140]  lr: 1.0000e-06  eta: 0:36:46  time: 0.8530  data_time: 0.0228  memory: 8439  loss: 1.5071

/home/vessel/anaconda3/envs/openmmlab/lib/python3.8/site-packages/mmengine/hooks/early_stopping_hook.py:148: UserWarning: Skip early stopping process since the evaluation results (dict_keys([])) do not include `monitor` (coco/bbox_mAP_50).
  warnings.warn(


08/16 09:52:23 - mmengine - INFO - Epoch(train)  [4][  1/140]  lr: 1.0000e-06  eta: 0:33:31  time: 0.7073  data_time: 0.0232  memory: 8439  loss: 1.5097  loss_cls: 0.0024  loss_bbox: 0.7298  loss_bbox_rf: 0.7774
08/16 09:52:24 - mmengine - INFO - Epoch(train)  [4][  2/140]  lr: 1.0000e-06  eta: 0:33:28  time: 0.7023  data_time: 0.0231  memory: 8439  loss: 1.4573  loss_cls: 0.0024  loss_bbox: 0.7041  loss_bbox_rf: 0.7507
08/16 09:52:24 - mmengine - INFO - Epoch(train)  [4][  3/140]  lr: 1.0000e-06  eta: 0:33:26  time: 0.6959  data_time: 0.0230  memory: 8439  loss: 1.4580  loss_cls: 0.0025  loss_bbox: 0.7046  loss_bbox_rf: 0.7509
08/16 09:52:25 - mmengine - INFO - Epoch(train)  [4][  4/140]  lr: 1.0000e-06  eta: 0:33:24  time: 0.6911  data_time: 0.0230  memory: 8439  loss: 1.4755  loss_cls: 0.0025  loss_bbox: 0.7131  loss_bbox_rf: 0.7599
08/16 09:52:25 - mmengine - INFO - Epoch(train)  [4][  5/140]  lr: 1.0000e-06  eta: 0:33:21  time: 0.6861  data_time: 0.0230  memory: 8439  loss: 1.4768

#### RUNNER TEST START

In [ ]:
runner.test_evaluator.metrics.append(DumpDetResults(out_file_path=f'{workdir}/test_result/test.pkl'))
# start testing
output_test_data =runner.test()

# Specify the file name
file_name = f'{workdir}/test_result/coco_metrics.json'# Specify the filepath
# Write the dictionary to a JSON file
with open(file_name, 'w') as json_file:
    json.dump(output_test_data, json_file, indent=4)

print(f"Data has been saved to {file_name}")